# 04 – Modeling & Submission

**Colab:** [Notebook in Colab öffnen](https://colab.research.google.com/github/jspldrch/DataMining_Final-Project/blob/main/notebooks/04_modeling.ipynb) → **Runtime: CPU** → alle Zellen ausführen.

Voraussetzung: Notebook **03** hat `outputs/processed/*.parquet` erzeugt (liegt auf Drive).

1. Baselines (Persistence, Regional-Median)
2. LightGBM
3. `outputs/submissions/submission_full.csv`

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

REPO_URL = "https://github.com/jspldrch/DataMining_Final-Project.git"
REPO_DIR = Path("/content/DataMining_Final-Project")
RANDOM_STATE = 42
VAL_FRAC = 0.2

if IS_COLAB:
    drive.mount("/content/drive")
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
        check=True,
    )
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))
    from scripts.colab_setup import init_environment

    env = init_environment(install_deps=False, skip_mount=True, skip_git=True)
else:
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / "scripts").exists() and (REPO_DIR.parent / "scripts").exists():
        REPO_DIR = REPO_DIR.parent
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))
    from scripts.colab_setup import init_environment

    env = init_environment(install_deps=True, skip_mount=True, skip_git=True)

PROCESSED_DIR = env["processed_dir"]
SUBMISSION_DIR = env["submission_dir"]
MODE = env["mode"]

_spec = importlib.util.spec_from_file_location(
    "dm_features", env["project_root"] / "scripts" / "features.py"
)
_features = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_features)
feature_columns = _features.feature_columns

path_train = PROCESSED_DIR / "train_labeled.parquet"
path_test = PROCESSED_DIR / "test_features.parquet"
if not path_train.exists():
    raise FileNotFoundError(f"{path_train} fehlt — zuerst 03_preprocessing ausführen.")

train_df = pd.read_parquet(path_train)
test_df = pd.read_parquet(path_test)
print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | Regionen: {train_df['region_id'].nunique()}")

In [ ]:
def regional_time_split(df, val_frac=0.2):
    train_parts, val_parts = [], []
    for _, g in df.groupby("region_id", sort=False):
        g = g.sort_values("ordinal")
        n_val = max(1, int(len(g) * val_frac))
        val_parts.append(g.iloc[-n_val:])
        train_parts.append(g.iloc[:-n_val])
    return pd.concat(train_parts), pd.concat(val_parts)


def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    print(f"{name:28s}  MAE={mae:.4f}  RMSE={rmse:.4f}")


tr, va = regional_time_split(train_df, VAL_FRAC)
y_va = va["score"]

if "score_persist7" in va.columns:
    evaluate(y_va, va["score_persist7"].fillna(0).clip(0, 5), "Persistence (7d)")
region_median = tr.groupby("region_id")["score"].median()
evaluate(y_va, va["region_id"].map(region_median).fillna(tr["score"].median()).clip(0, 5), "Regional median")

In [ ]:
FEATURES = [c for c in feature_columns() if c in train_df.columns]
X_tr, y_tr = tr[FEATURES].copy(), tr["score"]
X_va, y_va = va[FEATURES].copy(), va["score"]
X_tr["region_id"] = X_tr["region_id"].astype("category")
X_va["region_id"] = X_va["region_id"].astype("category")

model = lgb.LGBMRegressor(
    objective="regression", metric="mae", n_estimators=800, learning_rate=0.05,
    num_leaves=63, min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="mae",
          callbacks=[lgb.early_stopping(50, verbose=False)], categorical_feature=["region_id"])
evaluate(y_va, np.clip(model.predict(X_va), 0, 5), "LightGBM")

In [ ]:
X_full = train_df[FEATURES].copy()
X_full["region_id"] = X_full["region_id"].astype("category")
final_model = lgb.LGBMRegressor(
    objective="regression", n_estimators=model.best_iteration_ or 800,
    learning_rate=0.05, num_leaves=63, min_child_samples=50, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
final_model.fit(X_full, train_df["score"], categorical_feature=["region_id"])

X_test = test_df[FEATURES].copy()
X_test["region_id"] = X_test["region_id"].astype("category")
test_pred = np.clip(final_model.predict(X_test), 0, 5)

submission = test_df[["region_id", "date"]].copy()
submission["score"] = test_pred
out_path = SUBMISSION_DIR / f"submission_{MODE}.csv"
submission.to_csv(out_path, index=False)
print(f"Gespeichert: {out_path} ({len(submission):,} Zeilen)")
submission.head()